In [ ]:
import sys
print(sys.executable)

In [ ]:
!{sys.executable} -m pip install gdown numpy torch torchvision scikit-learn optuna timm huggingface_hub matplotlib

In [ ]:
from sklearn import show_versions
show_versions()

In [ ]:
import os
from pathlib import Path
import glob
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torch.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import optuna
import time
sys.path.insert(0, os.getcwd())
import model_lib
import importlib
import json

In [ ]:
import gdown
# Run preprocessing.py first to create dataset/sleep_waveform.pt.


In [ ]:
class EEGDataset(Dataset):
    def __init__(self, pt_filepath):
        """
        Args:
            pt_filepath (str): Saved (N, 2, 30, 200) Sleep-EDF waveform patches.
        """
        print(f"Memory-mapping dataset from {pt_filepath}...")
        # Tensors and ordinary string/list metadata support weights_only=True.
        checkpoint = torch.load(pt_filepath, map_location="cpu", weights_only=True, mmap=True)

        self.data = checkpoint['patches']
        # Keep test participants separate; map development indices without copying EEG.
        self.indices = np.flatnonzero(np.asarray(checkpoint['splits']) == 'development')
        self.labels = checkpoint['labels'][self.indices]
        self.groups = np.asarray(checkpoint['groups'])[self.indices]

        self.channels = checkpoint['channels']
        self.sfreq = checkpoint['sfreq']
        self.amplitude = checkpoint['preprocessing']['amplitude']
        self.class_to_idx = checkpoint['class_map']
        self.idx_to_class = {value: key for key, value in self.class_to_idx.items()}

        print("Mapped source shape:", tuple(self.data.shape))
        print("Input per epoch:", tuple(self.data.shape[1:]), "= channels, seconds, samples/second")
        print("Channels:", self.channels, "| Sampling rate:", self.sfreq, "Hz | Scale:", self.amplitude)
        print("Development samples:", len(self.labels))
        print("Participants:", len(set(self.groups)))
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Preserve (channels, 30 one-second patches, 200 samples) and stored amplitude.
        x = self.data[self.indices[idx]].float()
        y = self.labels[idx].long()

        return x, y

def get_stats(subset):
    """Diagnostic mean/std per EEG channel, across epochs and waveform samples.

    Input batches: (B, C, 30, 200). Flatten only the two time axes to (B, C, 6000).
    These statistics are not applied to the pretrained CBraMod input: the export
    already uses microvolts / 100. Call on a training subset when inspecting data.
    """
    loader = DataLoader(subset, batch_size=32, shuffle=False, num_workers=0)
    sum_ = torch.zeros(subset[0][0].shape[0], dtype=torch.float64)
    sum_sq = torch.zeros_like(sum_)
    total_samples = 0

    for waveforms, _ in loader:
        waveforms = waveforms.double().flatten(start_dim=2)
        sum_ += waveforms.sum(dim=(0, 2))
        sum_sq += waveforms.square().sum(dim=(0, 2))
        total_samples += waveforms.shape[0] * waveforms.shape[2]

    mean = sum_ / total_samples
    variance = sum_sq / total_samples - mean.square()
    std = variance.clamp_min(0).sqrt()
    return mean.float(), std.float()

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_f1 = -float('inf')
        self.early_stop = False

    def __call__(self, val_f1):
        if val_f1 > self.best_f1 + self.min_delta:
            self.best_f1 = val_f1
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
class ApplyTransform(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

def calculate_metrics(targets, outputs):
    outputs = outputs.detach().float().cpu()
    targets = targets.view(-1).detach().cpu().numpy().astype(int)

    probs = torch.softmax(outputs, dim=1).numpy()
    preds = probs.argmax(axis=1)

    acc = accuracy_score(targets, preds)
    auroc = roc_auc_score(targets, probs, labels=range(5), multi_class='ovr', average='macro')
    prec = precision_score(targets, preds, labels=range(5), average='macro', zero_division=0)
    rec = recall_score(targets, preds, labels=range(5), average='macro', zero_division=0)
    f1 = f1_score(targets, preds, labels=range(5), average='macro', zero_division=0)

    kappa = cohen_kappa_score(targets, preds, labels=range(5))

    return acc, auroc, prec, rec, f1, kappa

def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer, scheduler, device, use_sched, use_es, max_epochs=50):
    # Development/inner validation only: this function selects an epoch using val_loader.
    scaler = GradScaler('cuda', enabled=device.type == 'cuda')
    early_stopping = EarlyStopping(patience=10, min_delta=0.001)

    # Select the checkpoint and stop training using validation Macro-F1.
    best_val_auroc = 0.0
    best_acc = 0.0
    best_prec = 0.0
    best_rec = 0.0
    best_f1 = -float('inf')
    best_kappa = 0.0
    best_epoch, best_state = 0, None
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    for epoch in range(max_epochs):
        epoch_start_time = time.time()

        model.train()
        train_loss = 0.0

        all_train_targets, all_train_outputs = [], []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device=device, dtype=torch.float32), labels.to(device=device, dtype=torch.long)

            optimizer.zero_grad()

            with autocast('cuda', enabled=device.type == 'cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * len(labels)

            all_train_targets.append(labels.detach().cpu())
            all_train_outputs.append(outputs.detach().cpu())

        model.eval()
        val_loss = 0.0
        all_targets, all_outputs = [], []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device=device, dtype=torch.float32), labels.to(device=device, dtype=torch.long)

                with autocast('cuda', enabled=device.type == 'cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                val_loss += loss.item() * len(labels)
                all_targets.append(labels.cpu())
                all_outputs.append(outputs.cpu())

        avg_train_loss = train_loss / len(train_loader.dataset)
        avg_val_loss = val_loss / len(val_loader.dataset)

        train_targets_tensor = torch.cat(all_train_targets)
        train_outputs_tensor = torch.cat(all_train_outputs)
        train_acc, train_auroc, train_prec, train_rec, train_f1, train_kappa = calculate_metrics(train_targets_tensor, train_outputs_tensor)

        targets_tensor = torch.cat(all_targets)
        outputs_tensor = torch.cat(all_outputs)
        acc, auroc, prec, rec, f1, kappa = calculate_metrics(targets_tensor, outputs_tensor)
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(acc)

        epoch_duration = time.time() - epoch_start_time

        print(f"Epoch {epoch+1:02d}/{max_epochs} | Time: {epoch_duration:.2f}s")
        print(f"  [Train] Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | AUROC: {train_auroc:.4f} | Prec: {train_prec:.4f} | Rec: {train_rec:.4f} | F1: {train_f1:.4f} | Kappa: {train_kappa:.4f}")
        print(f"  [Val]   Loss: {avg_val_loss:.4f} | Acc: {acc:.4f} | AUROC: {auroc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | Kappa: {kappa:.4f}")

        if use_sched:
            scheduler.step()

        if f1 > best_f1:
            best_val_auroc = auroc
            best_acc = acc
            best_prec = prec
            best_rec = rec
            best_f1 = f1
            best_kappa = kappa
            best_epoch = epoch + 1
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_targets = targets_tensor.numpy().astype(int)
            best_preds = outputs_tensor.float().argmax(dim=1).numpy()

        if use_es:
            early_stopping(f1)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}!")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.eval()
        print(f'Restored best validation-Macro-F1 weights from epoch {best_epoch}.')

    if best_state is not None:
        cm = confusion_matrix(best_targets, best_preds, labels=range(5))
        print(f'Validation EEG-epoch confusion matrix at epoch {best_epoch} (argmax over five stages)')
        print('Rows = true [W, N1, N2, N3, REM]; columns = predicted stages')
        print(cm)
        ConfusionMatrixDisplay(cm, display_labels=['W', 'N1', 'N2', 'N3', 'REM']).plot(cmap='Blues', values_format='d', colorbar=False)
        plt.title(f'Validation (EEG epochs) - best Macro-F1 epoch {best_epoch}')
        plt.tight_layout()
        plt.show()

        epochs = range(1, len(train_losses) + 1)
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        plt.plot(epochs, train_losses, label='Train Loss')
        plt.plot(epochs, val_losses, label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.subplot(1, 2, 2)
        plt.plot(epochs, train_accuracies, label='Train Accuracy')
        plt.plot(epochs, val_accuracies, label='Val Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        plt.suptitle('EEG-epoch metrics; training measured during updates')
        plt.tight_layout()
        plt.show()

    return best_val_auroc, best_acc, best_prec, best_rec, best_f1, best_kappa

def objective(trial, dataset, device):
    lr = trial.suggest_float("lr", 1e-5, 3e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    y = dataset.labels.numpy()
    groups = np.array(dataset.groups)

    train_idx, val_idx = next(sgkf.split(np.zeros(len(y)), y, groups))
    assert set(groups[train_idx]).isdisjoint(set(groups[val_idx]))

    train_subset_raw = Subset(dataset, train_idx)
    val_subset_raw = Subset(dataset, val_idx)

    # Already microvolts / 100, matching the CBraMod sleep loader; no extra z-score.
    normalization = None
    train_subset = ApplyTransform(train_subset_raw, transform=normalization)
    val_subset = ApplyTransform(val_subset_raw, transform=normalization)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, prefetch_factor=prefetch_factor,
                                num_workers=num_workers, pin_memory=pin_memory, persistent_workers=persistent_workers)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, prefetch_factor=prefetch_factor,
                                num_workers=num_workers, pin_memory=pin_memory, persistent_workers=persistent_workers)

    model = model_lib.CBraModClassifier(
        in_channels=dataset.data.shape[1], checkpoint_path=CHECKPOINT, freeze_encoder=False,
    ).to(device)
    # model = torch.compile(model)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=100,
        eta_min=1e-6
    )

    best_auroc, best_acc, best_prec, best_rec, best_f1, best_kappa = train_and_evaluate(
        model, train_loader, val_loader, criterion, optimizer, scheduler, device, use_sched=True, use_es=True, max_epochs=100
    )

    return best_f1

In [ ]:
# Run from the directory containing this notebook and model_lib.py.
PROJECT_DIR = os.getcwd()

DATA_DIR = os.path.join(PROJECT_DIR, "dataset")
MODEL_DIR = os.path.join(PROJECT_DIR, "spec_model")

DATA_PATH = os.path.join(DATA_DIR, "sleep_waveform_small.pt")
CHECKPOINT = os.path.join(PROJECT_DIR, "pretrained_weights", "pretrained_weights.pth")

# Create model directory if it doesn't exist
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)

dataset = EEGDataset(pt_filepath=DATA_PATH)
print(f"Found {len(dataset)} total samples across {len(dataset.class_to_idx)} classes.")

In [ ]:
importlib.reload(model_lib)

lr = 3e-5
batch_size = 32
weight_decay = 1e-3
n_splits = 5
max_epochs = 80

# Linux/Colab/Vast workers. For a macOS notebook use 0 / None / False below.
num_workers = 4
prefetch_factor = 2
persistent_workers = True
pin_memory = True

# Regenerates development folds: use the same split for the CNN comparison.
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
y = dataset.labels.numpy()
groups = np.array(dataset.groups)

cv_metrics = {
    'auroc': [],
    'acc': [],
    'prec': [],
    'rec': [],
    'f1': [],
    'kappa': []
}

for fold, (train_idx, val_idx) in enumerate(sgkf.split(np.zeros(len(y)), y, groups)):
    if fold > 1: break  # chỉ train và đánh giá Fold 1 2
    print(f"\n{'='*20} Fold {fold + 1}/{n_splits} {'='*20}")

    train_subset_raw = Subset(dataset, train_idx)
    val_subset_raw = Subset(dataset, val_idx)

    # The saved waveform is already microvolts / 100. Do not z-score or divide again.
    # Optional diagnostics only: mean, std = get_stats(train_subset_raw)
    # Reference: https://github.com/wjq-learning/CBraMod/blob/main/datasets/isruc_dataset.py
    normalization = None

    train_subset = ApplyTransform(train_subset_raw, transform=normalization)
    val_subset = ApplyTransform(val_subset_raw, transform=normalization)

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, prefetch_factor=prefetch_factor,
                                num_workers=num_workers, pin_memory=pin_memory, persistent_workers=persistent_workers)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, prefetch_factor=prefetch_factor,
                                num_workers=num_workers, pin_memory=pin_memory, persistent_workers=persistent_workers)

    model = model_lib.CBraModClassifier(
        in_channels=dataset.data.shape[1],
        checkpoint_path=CHECKPOINT,
        freeze_encoder=False,
    ).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=100,
        eta_min=1e-6
    )

    best_auroc, best_acc, best_prec, best_rec, best_f1, best_kappa = train_and_evaluate(
        model, train_loader, val_loader, criterion, optimizer, scheduler, device, use_sched=True, use_es=True, max_epochs=max_epochs
    )

    cv_metrics['auroc'].append(best_auroc)
    cv_metrics['acc'].append(best_acc)
    cv_metrics['prec'].append(best_prec)
    cv_metrics['rec'].append(best_rec)
    cv_metrics['f1'].append(best_f1)
    cv_metrics['kappa'].append(best_kappa)

    print(f"\n--- Fold {fold + 1} Metrics at Best Validation Macro-F1 ---")
    print(f"AUROC: {best_auroc:.4f} | Acc: {best_acc:.4f} | Prec: {best_prec:.4f} | Rec: {best_rec:.4f} | F1: {best_f1:.4f} | Kappa: {best_kappa:.4f}")

print(f"\n{'='*20} Final CV Results ({len(cv_metrics['acc'])}/{n_splits} folds completed) {'='*20}")
for metric_name, values in cv_metrics.items():
    mean_val = np.mean(values)
    std_val = np.std(values)

    display_name = metric_name.upper() if metric_name != 'acc' and metric_name != 'prec' and metric_name != 'rec' else metric_name.capitalize()

    print(f"Average {display_name}: {mean_val:.4f} ± {std_val:.4f}")